# Lecture 05 — JavaScript-rendered pages with Playwright

> *"If `view-source:` shows you nothing, you're not on the wrong site, you're using the wrong tool."*

In Lecture 00 we drew the line between **server-rendered** pages (data is in the HTML) and **client-rendered** pages (JS fills in an empty `<div id="root">`). For the latter, `httpx` + BeautifulSoup is not enough — there's nothing to parse. You need something that runs JavaScript.

That used to be PyQt5 + `QtWebEngine` (the old version of this tutorial), or Selenium, or Puppeteer. In 2026 the practical answer is **Playwright**: maintained by Microsoft, runs Chromium / Firefox / WebKit, has a clean Python API, and works in both sync and async modes.

## What you'll be able to do after this lecture

- Install Playwright and a browser engine.
- Open a page, wait for it to finish rendering, and grab the resulting HTML.
- Click buttons, scroll, and fill in forms when needed.
- Capture network requests (the trick that often makes scraping unnecessary).
- Decide whether you need a browser at all, or whether the API exists.

## Setup

```bash
pip install playwright beautifulsoup4 httpx
python -m playwright install chromium
```

That second line downloads a Chromium build (~150 MB). Do it once.


## 1. Hello, browser

In [ ]:
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page()
    page.goto("https://quotes.toscrape.com/js/")  # JS-rendered version of quotes.toscrape
    html = page.content()
    print("length:", len(html))
    print(html[:300])
    browser.close()


`quotes.toscrape.com/js/` is the same data as the static version (`/`), but rendered in the browser. If you `httpx.get()` it directly you'll see an empty page — try it. With Playwright, the HTML you get is *post-render*, just like what you'd see in your browser.

That's the whole magic. Everything else in this lecture is technique.

## 2. Sync vs async API

Playwright has two flavors: `playwright.sync_api` and `playwright.async_api`. They have the same surface, just sync or async-style. We'll use the sync API in this lecture for clarity. In Lecture 06 we'll switch to async.

## 3. Wait for the right thing

Pages don't finish loading at one moment. There are several "done" signals, in increasing order of patience:

| Signal              | When it fires                                           | Use when                              |
|---------------------|---------------------------------------------------------|---------------------------------------|
| `load`              | `window.load` fires (DOM + sub-resources loaded)        | Static-ish pages                      |
| `domcontentloaded`  | DOM is parsed (sub-resources may still be loading)      | You only need DOM, faster             |
| `networkidle`       | No network requests for 500ms                           | Heavy SPAs (slow but sure)            |
| `wait_for_selector` | A specific element appears                              | You know what you're waiting for      |

`wait_for_selector` is almost always the right answer. Don't sleep for 5 seconds and hope — wait for the element you actually need.

In [ ]:
from playwright.sync_api import sync_playwright

URL = "https://quotes.toscrape.com/js/"

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page()
    page.goto(URL)

    # Wait for the first quote element to render
    page.wait_for_selector(".quote")

    quotes = page.query_selector_all(".quote")
    print(f"found {len(quotes)} quotes")
    for q in quotes[:3]:
        text = q.query_selector(".text").inner_text()
        author = q.query_selector(".author").inner_text()
        print(f"  {author}: {text[:60]}...")

    browser.close()


## 4. The pattern: render with Playwright, parse with BeautifulSoup

Playwright has its own DOM API (`query_selector_all`, `inner_text`), and it's fine. But BeautifulSoup is more familiar by now, and using it everywhere keeps your codebase uniform.

The pattern: use Playwright *only* to render. Grab `page.content()` (a string of HTML), feed it to BeautifulSoup, and parse exactly like you would with a static page.

In [ ]:
from bs4 import BeautifulSoup

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page()
    page.goto(URL)
    page.wait_for_selector(".quote")
    html = page.content()
    browser.close()

soup = BeautifulSoup(html, "lxml")
for q in soup.select(".quote")[:3]:
    print(q.select_one(".author").get_text(strip=True), "->", q.select_one(".text").get_text(strip=True)[:60])


**Once the HTML is in BeautifulSoup, all of Lecture 03 applies.** Playwright was just the rendering step.

## 5. Pagination via clicking

Many SPAs don't have URL-based pagination — they have a "Load more" button or infinite scroll. Playwright handles both.

### Clicking a button

In [ ]:
# quotes.toscrape.com/js/ has page-number links rather than a Load More button.
# Same idea: navigate to the next page until none.

def collect_all_quotes(start_url: str) -> list[dict]:
    quotes = []
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        url = start_url
        while True:
            page.goto(url)
            page.wait_for_selector(".quote")
            soup = BeautifulSoup(page.content(), "lxml")
            for q in soup.select(".quote"):
                quotes.append({
                    "text": q.select_one(".text").get_text(strip=True),
                    "author": q.select_one(".author").get_text(strip=True),
                    "tags": [t.get_text(strip=True) for t in q.select(".tag")],
                })
            next_link = soup.select_one("li.next a")
            if next_link is None:
                break
            from urllib.parse import urljoin
            url = urljoin(start_url, next_link["href"])
        browser.close()
    return quotes


# all_quotes = collect_all_quotes("https://quotes.toscrape.com/js/")
# print(len(all_quotes))


### Infinite scroll

For pages that load more content as you scroll, scroll the page in a loop and wait for new content.

In [ ]:
def scroll_to_bottom(page, item_selector: str, max_scrolls: int = 30, settle_ms: int = 500):
    """Scroll until no new items appear (or max_scrolls reached)."""
    last_count = 0
    for _ in range(max_scrolls):
        page.mouse.wheel(0, 5000)
        page.wait_for_timeout(settle_ms)
        count = len(page.query_selector_all(item_selector))
        if count == last_count:
            break
        last_count = count
    return last_count


## 6. Filling forms and clicking through login walls

If the data you want is behind a login, Playwright can fill the form. **A note before doing this:** check the site's ToS and your own ethics. Logged-in scraping has very different legal weight than public scraping.

```python
page.goto("https://example.com/login")
page.fill("input[name='username']", "alice")
page.fill("input[name='password']", os.environ["SITE_PW"])
page.click("button[type='submit']")
page.wait_for_url("https://example.com/dashboard")
```

Best practice: never put credentials in code. Always read from env vars or a secret manager.

## 7. Capturing network requests — the most useful trick

This is the lecture's secret weapon. Playwright lets you eavesdrop on every HTTP request the page makes. Often the data you wanted is in a single XHR call returning clean JSON, which means:

1. You watch the page once with Playwright.
2. You note the JSON URL.
3. You make subsequent requests directly with `httpx`. **No browser needed.**

10x speedup, half the dependencies, more reliable code.

In [ ]:
def show_xhr_requests(url: str, limit: int = 20):
    seen = []
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        context = browser.new_context()
        page = context.new_page()

        def on_response(response):
            ct = response.headers.get("content-type", "")
            if "json" in ct or response.request.resource_type in ("xhr", "fetch"):
                seen.append((response.status, response.request.method, response.url))

        page.on("response", on_response)
        page.goto(url, wait_until="networkidle")
        browser.close()
    for s, m, u in seen[:limit]:
        print(f"  {m} {s}  {u}")


# show_xhr_requests("https://quotes.toscrape.com/js/")


Run that against any SPA you're curious about. You'll often see exactly the API endpoint that delivers the data, complete with query parameters. Lecture 07 turns this into a methodology.

## 8. Headed vs headless, debugging

Set `headless=False` to actually see the browser window pop up. Combine with `slow_mo=200` (ms between actions) to watch your code as it runs. Invaluable when something isn't working.

In [ ]:
# For debugging only — uncomment and run interactively
# with sync_playwright() as p:
#     browser = p.chromium.launch(headless=False, slow_mo=300)
#     page = browser.new_page()
#     page.goto("https://quotes.toscrape.com/js/")
#     page.wait_for_selector(".quote")
#     page.screenshot(path="/tmp/quotes.png")
#     browser.close()


Other useful debugging tools:
- **`page.screenshot(path=...)`** — capture what the browser sees right now.
- **`page.pause()`** — opens the Playwright Inspector, a step-debugger.
- **`PWDEBUG=1 python yourscript.py`** — same effect from outside the script.


## 9. Cost-benefit — when *not* to use Playwright

A real Chromium uses ~200 MB of RAM per page and takes ~1 second to spin up. That's expensive. Don't use it when you don't have to.

| Situation                                   | Tool                       |
|---------------------------------------------|----------------------------|
| Static HTML you can `httpx.get`             | `httpx` only (Lec 02-04)   |
| SPA but XHR endpoint is discoverable        | `httpx` against the API (Lec 07) |
| Truly opaque SPA, must execute JS to render | Playwright                 |
| Login wall + JS-only flow                   | Playwright                 |
| Site with hard anti-bot (Cloudflare etc.)   | Playwright + stealth, or **don't** (Lec 08) |

The right answer for many SPAs is *not* "drive a browser forever". It's "use Playwright **once** to find the API, then `httpx` for the actual crawl".


## Recap

- Server-rendered pages don't need a browser. Client-rendered pages do.
- Playwright is the modern answer: clean API, fast, runs all three engines.
- Always `wait_for_selector` for *the thing you need*, not a fixed sleep.
- Capture HTML via `page.content()` and parse with BeautifulSoup — keeps your codebase uniform.
- Watching the Network tab via Playwright's response listener is the fastest way to find a hidden API.

## Exercises

1. Use Playwright to load `https://quotes.toscrape.com/js-delayed/` (loads with a 10-second delay). Get the quotes anyway.
2. Modify `collect_all_quotes` to deduplicate by `(author, text[:50])` and also save author URLs by clicking through.
3. Pick a real SPA you visit (Twitter, Bluesky, a streaming service's catalog). Use the network-listener trick to find the API endpoint that returns the data you'd want. Note the URL and the request shape — but don't actually scrape it without checking ToS.
4. Compare time-to-page for the same content fetched three ways: `httpx`, Playwright with `networkidle`, Playwright with `wait_for_selector` for a specific element. Which is fastest? Why?

## Up next

**Lecture 06** — both `httpx` and Playwright support `async`/`await`. Async crawling is how you go from one page per second to fifty without melting the target server. We'll cover semaphores, per-host rate limiting, and exponential backoff in their natural async habitat.
